# 03. Rule-based 통계 기반 예측

**목적**: 상관계수 기반 가중치를 활용한 Rule-based 위험도 예측 방법론을 설명합니다.

## 왜 Rule-based인가?

ML 모델(Ridge, XGBoost, CatBoost)의 R² 값이 0.3 이하로 낮아,
**상관계수 기반의 해석 가능한 Rule-based 방식**으로 전환했습니다.

## 1. 상관계수 기반 전역 가중치

global_corr.ipynb에서 계산된 death_rate와의 상관계수를 정규화하여 가중치를 산출했습니다.

In [1]:
import pandas as pd
import numpy as np

# 전역 가중치 (global_corr.ipynb 결과)
GLOBAL_WEIGHTS = {
    "single_household_ratio": 0.368467,       # 1인가구 비율
    "aging_index": 0.311710,                  # 노령화지수
    "elderly_population_ratio": 0.185376,     # 65+ 인구 비율
    "low_income_elderly_80_over_ratio": 0.095469,  # 저소득 80+
    "low_income_elderly_65_79_ratio": 0.038977,    # 저소득 65~79
}

print("📊 전역 가중치 (중요도 순)")
print("="*50)
for feat, weight in sorted(GLOBAL_WEIGHTS.items(), key=lambda x: -x[1]):
    bar = "█" * int(weight * 30)
    print(f"{feat:40s} {weight:.4f} {bar}")

📊 전역 가중치 (중요도 순)
single_household_ratio                   0.3685 ███████████
aging_index                              0.3117 █████████
elderly_population_ratio                 0.1854 █████
low_income_elderly_80_over_ratio         0.0955 ██
low_income_elderly_65_79_ratio           0.0390 █


## 2. 정규화 함수

각 피처를 0~1 범위로 정규화하여 가중합 계산에 사용합니다.

In [2]:
def normalize(value: float, max_value: float) -> float:
    """
    값을 0~1 범위로 정규화
    """
    if max_value <= 0:
        return 0.0
    x = value / max_value
    return max(0.0, min(1.0, x))  # 0~1 클리핑

# 각 피처별 최대값 (정규화 기준)
MAX_VALUES = {
    "single_household_ratio": 40.0,              # 최대 40%
    "low_income_elderly_65_79_ratio": 30.0,      # 최대 30%
    "low_income_elderly_80_over_ratio": 30.0,    # 최대 30%
    "aging_index": 250.0,                        # 최대 250
    "elderly_population_ratio": 35.0,            # 최대 35%
}

print("정규화 기준 최대값:")
for feat, max_val in MAX_VALUES.items():
    print(f"  {feat}: {max_val}")

정규화 기준 최대값:
  single_household_ratio: 40.0
  low_income_elderly_65_79_ratio: 30.0
  low_income_elderly_80_over_ratio: 30.0
  aging_index: 250.0
  elderly_population_ratio: 35.0


## 3. 위험지수 계산 공식

**Risk Score = Σ (weight_i × normalized_value_i)**

- 각 피처를 정규화하여 0~1 범위로 변환
- 가중치를 곱하여 합산
- 최종 Risk Score: 0 (저위험) ~ 1 (고위험)

In [3]:
def compute_risk_score(features: dict, weights: dict = GLOBAL_WEIGHTS) -> float:
    """
    구별 위험지수 계산 (0~1)
    
    Parameters:
    - features: {feature_name: value} 딕셔너리
    - weights: 가중치 딕셔너리
    
    Returns:
    - risk_score: 0~1 사이 값
    """
    # 1) 피처 정규화
    scores = {}
    for feat, value in features.items():
        max_val = MAX_VALUES.get(feat, 100.0)
        scores[feat] = normalize(value, max_val)
    
    # 2) 가중합 계산
    risk = 0.0
    for feat, score in scores.items():
        w = weights.get(feat, 0.0)
        risk += w * score
    
    # 3) 0~1 클리핑
    return max(0.0, min(1.0, risk))

def to_risk_level(score: float) -> str:
    """
    위험등급 분류
    """
    if score >= 0.7:
        return "🔴 HIGH"
    elif score >= 0.4:
        return "🟠 MEDIUM"
    else:
        return "🟢 LOW"

print("위험지수 계산 함수 정의 완료!")

위험지수 계산 함수 정의 완료!


## 4. 예시: 구별 위험지수 계산

In [4]:
# 예시 데이터 (2023년 기준 가상 데이터)
sample_regions = {
    "강남구": {
        "single_household_ratio": 32.5,
        "low_income_elderly_65_79_ratio": 8.2,
        "low_income_elderly_80_over_ratio": 12.3,
        "aging_index": 145.6,
        "elderly_population_ratio": 18.2,
    },
    "관악구": {
        "single_household_ratio": 38.2,
        "low_income_elderly_65_79_ratio": 15.5,
        "low_income_elderly_80_over_ratio": 18.7,
        "aging_index": 198.3,
        "elderly_population_ratio": 24.1,
    },
    "송파구": {
        "single_household_ratio": 28.7,
        "low_income_elderly_65_79_ratio": 6.8,
        "low_income_elderly_80_over_ratio": 9.5,
        "aging_index": 112.4,
        "elderly_population_ratio": 15.3,
    },
}

print("\n" + "="*60)
print("구별 위험지수 계산 결과")
print("="*60)

results = []
for region, features in sample_regions.items():
    score = compute_risk_score(features)
    level = to_risk_level(score)
    results.append({"구": region, "위험지수": score, "등급": level})
    print(f"{region:8s} | 위험지수: {score:.4f} | {level}")

print("="*60)


구별 위험지수 계산 결과
강남구      | 위험지수: 0.6271 | 🟠 MEDIUM
관악구      | 위험지수: 0.8064 | 🔴 HIGH
송파구      | 위험지수: 0.5246 | 🟠 MEDIUM


## 5. 구별 가중치 조정

global_corr.ipynb에서는 **구별로 상관계수를 다르게 계산**하여
상위 2개 피처는 가중치 1.2배, 하위 1개 피처는 0.8배로 조정했습니다.

In [5]:
# region_feature_weights.csv 예시
region_weights_example = pd.DataFrame([
    {"region": "강남구", "feature": "single_household_ratio", "weight": 0.372259},
    {"region": "강남구", "feature": "low_income_elderly_65_79_ratio", "weight": 0.047254},
    {"region": "강남구", "feature": "low_income_elderly_80_over_ratio", "weight": 0.115742},
    {"region": "강남구", "feature": "aging_index", "weight": 0.314918},
    {"region": "강남구", "feature": "elderly_population_ratio", "weight": 0.149827},
])

print("강남구 조정된 가중치:")
print(region_weights_example.to_string(index=False))

강남구 조정된 가중치:
region                          feature   weight
   강남구           single_household_ratio 0.372259
   강남구   low_income_elderly_65_79_ratio 0.047254
   강남구 low_income_elderly_80_over_ratio 0.115742
   강남구                      aging_index 0.314918
   강남구         elderly_population_ratio 0.149827


## 6. 미래 예측 방법

### 예측 파이프라인

```
1. 2023년 실측 고독사 건수 (target_value)
2. 예측 연도별 노인 인구 (ELDERLY_HISTORY)
3. 2023년 통계 지표 유지 가정
4. 예측 고독사 = 2023 고독사 × (미래 노인인구 / 2023 노인인구)
5. 연간 ±15% 변동 제한 (smooth change)
```

In [6]:
# 미래 예측 예시
def predict_future(base_year_deaths: float, 
                   base_population: float,
                   future_population: float,
                   max_change: float = 0.15) -> float:
    """
    미래 고독사 예측
    
    Parameters:
    - base_year_deaths: 기준연도 고독사 수
    - base_population: 기준연도 노인 인구
    - future_population: 예측연도 노인 인구
    - max_change: 최대 연간 변동률 (기본 15%)
    """
    # 인구 비율 기반 예측
    ratio = future_population / base_population
    raw_pred = base_year_deaths * ratio
    
    # 급격한 변동 제한
    lower = base_year_deaths * (1 - max_change)
    upper = base_year_deaths * (1 + max_change)
    
    return max(lower, min(upper, raw_pred))

# 예시: 강남구 2024년 예측
gangnam_2023_deaths = 127
gangnam_2023_pop = 85432
gangnam_2024_pop = 87654  # 통계청 예측

gangnam_2024_pred = predict_future(gangnam_2023_deaths, gangnam_2023_pop, gangnam_2024_pop)
print(f"\n강남구 2024년 예측 고독사 건수: {gangnam_2024_pred:.0f}건")


강남구 2024년 예측 고독사 건수: 130건


## 7. 결론

### Rule-based 방식의 장점

| 항목 | ML 모델 | Rule-based |
|------|---------|------------|
| 해석 가능성 | 낮음 | ✅ 높음 |
| 소규모 데이터 | 과적합 위험 | ✅ 안정적 |
| 도메인 지식 | 미반영 | ✅ 반영 가능 |
| R² 의존 | 높음 | ✅ 불필요 |

### 핵심 가중치 (death_rate 상관계수 기반)

1. **1인가구 비율** (0.368) - 가장 강력한 예측 변수
2. **노령화지수** (0.312) - 인구 구조 반영
3. **노인 인구 비율** (0.185) - 지역 특성
4. **저소득 80+ 비율** (0.095) - 취약계층
5. **저소득 65-79 비율** (0.039) - 보조 지표

In [7]:
print("\n" + "="*50)
print("✅ Rule-based 통계 기반 예측 분석 완료")
print("="*50)


✅ Rule-based 통계 기반 예측 분석 완료
